In [ ]:
import ROOT
import numpy as np
import matplotlib.pyplot as plt
import os
import pandas as pd
%matplotlib inline 

## Loading the file

In [ ]:
datatype = "2012"
eventtype = "23903003"
polarity = "magdown"
sign = "ws"

In [ ]:
## Should get this from the config
#data_prefix = "root://eoslhcb.cern.ch//eos/lhcb/wg/semileptonic/RDsHad/AP/pidgen_merged/"
data_prefix = "root://eoslhcb.cern.ch//eos/lhcb/user/b/bcouturi/rds/pidgen_merged/"

filename =  data_prefix + f"rds_categ_{datatype}_{eventtype}_{polarity}_{sign}.root"
metadata_filename =  data_prefix + f"rds_categ_metadata_{datatype}_{eventtype}_{polarity}_{sign}.json"
metadata_local_filename = os.path.basename(metadata_filename).rsplit("?", 1)[0]
print(metadata_local_filename)
print(f"Loading DecayTree from {filename}")
rdf = ROOT.RDataFrame("DecayTree", [ filename ])
print(f"Number of candidates: {rdf.Count().GetValue()}")

In [ ]:
!pwd

In [ ]:
import json
from XRootD import client
with client.File() as f:
    f.open(metadata_filename)
    buffer = f.read()
metadata = json.loads(buffer[1])
detailed_categories_map = metadata["detailed_categories"]
simple_categories_map = metadata["simple_categories"]

In [ ]:
def mygroupby(d, groupbycols):
    g = d.groupby(groupbycols).size().reset_index(name='count').sort_values([ 'count'], ascending=False).reset_index(drop=True)
    g["Percentage"] = g.apply(lambda row: 100 * row["count"]/d.shape[0], axis=1)
    g["cumulative %"] = g["Percentage"].cumsum(axis = 0)
    return g

## Sample Statistics 

In [ ]:
filter_vars = [ "simplified", "category"]
plot_vars = [ "B_Y_SEP", "B_M", "Y_M", "q2_2", "tauY_2" ]

In [ ]:
df = pd.DataFrame(rdf.Cache(plot_vars + filter_vars).AsNumpy())
df["category name"] = df.apply(lambda row: detailed_categories_map[str(int(row['category']))], axis=1)
df["simplified category"] = df.apply(lambda row: simple_categories_map[str(int(row['simplified']))], axis=1)
  

In [ ]:
g = mygroupby(df, ['category name'])
g        

In [ ]:
gs = mygroupby(df, ['simplified category'])
gs